# Исследование рынка видеоигр в 2000–2013 годах

**Автор:** Ефимов Алексей  
**Проект:** «Секреты Темнолесья», спринт 7

## Цель проекта

Подготовить исторические данные о рынке видеоигр для дальнейшего исследования развития игровой индустрии в период с 2000 по 2013 год.

## Задачи

1. Загрузить данные и изучить их структуру.
2. Привести названия столбцов к `snake_case`.
3. Проверить и скорректировать типы данных.
4. Изучить и обработать пропуски.
5. Найти явные и неявные дубликаты.
6. Отобрать игры, выпущенные с 2000 по 2013 год включительно.
7. Категоризовать игры по оценкам пользователей и критиков.
8. Выделить топ-7 платформ по количеству выпущенных игр.

## Описание данных

- `Name` — название игры;
- `Platform` — игровая платформа;
- `Year of Release` — год выпуска;
- `Genre` — жанр;
- `NA sales`, `EU sales`, `JP sales`, `Other sales` — продажи в регионах, млн копий;
- `Critic Score` — оценка критиков от 0 до 100;
- `User Score` — оценка пользователей от 0 до 10;
- `Rating` — возрастной рейтинг ESRB.

## 1. Загрузка и знакомство с данными

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
# На платформе Практикума используется первый путь, локально — второй.
possible_paths = [Path('/datasets/new_games.csv'), Path('new_games.csv')]
data_path = next((path for path in possible_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        'Файл new_games.csv не найден. Поместите его рядом с тетрадью.'
    )

df = pd.read_csv(data_path)
print(f'Данные загружены из: {data_path}')

Данные загружены из: new_games.csv


In [3]:
df.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.00,Sports,41.36,28.96,3.77,8.45,76.00,8,E
1,Super Mario Bros.,NES,1985.00,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.00,Racing,15.68,12.76,3.79,3.29,82.00,8.3,E
3,Wii Sports Resort,Wii,2009.00,Sports,15.61,10.93,3.28,2.95,80.00,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.00,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


In [5]:
print(f'Количество строк: {df.shape[0]}')
print(f'Количество столбцов: {df.shape[1]}')
print('\nНазвания столбцов:')
print(df.columns.tolist())

Количество строк: 16956
Количество столбцов: 11

Названия столбцов:
['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales', 'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating']


In [6]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Name,16954,11559,Need for Speed: Most Wanted,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Platform,16956,31,PS2,2189,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Year of Release,16681.00,NaN,NaN,NaN,2006.49,5.87,1980.00,2003.00,2007.00,2010.00,2016.00
Genre,16954,24,Action,3405,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NA sales,16956.00,NaN,NaN,NaN,0.26,0.81,0.00,0.00,0.08,0.24,41.36
EU sales,16956,308,0.0,5950,NaN,NaN,NaN,NaN,NaN,NaN,NaN
JP sales,16956,245,0.0,10680,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Other sales,16956.00,NaN,NaN,NaN,0.05,0.19,0.00,0.00,0.01,0.03,10.57
Critic Score,8242.00,NaN,NaN,NaN,68.93,13.94,13.00,60.00,71.00,79.00,98.00
User Score,10152,96,tbd,2464,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Промежуточный вывод

В исходном датасете 16 956 строк и 11 столбцов. Состав полей соответствует описанию. Уже при первичном знакомстве заметны проблемы, требующие предобработки:

- названия столбцов содержат заглавные буквы и пробелы;
- в названиях игр, жанрах, годах выпуска, оценках и рейтингах есть пропуски;
- `eu_sales`, `jp_sales` и `user_score` имеют тип `object`, хотя должны быть числовыми;
- `year_of_release` имеет вещественный тип из-за пропусков;
- категориальные значения нужно проверить на разные варианты регистра;
- данные необходимо проверить на явные дубликаты.

## 2. Проверка ошибок и предобработка данных

### 2.1. Названия столбцов

In [7]:
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

In [8]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', '_', regex=True)
)

df.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')

Названия столбцов приведены к стилю `snake_case`: строчные буквы, пробелы заменены подчёркиваниями.

### 2.2. Типы данных

In [9]:
df.dtypes

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales            object
jp_sales            object
other_sales        float64
critic_score       float64
user_score          object
rating              object
dtype: object

In [10]:
# Покажем строковые значения, мешающие преобразованию числовых столбцов.
for column in ['eu_sales', 'jp_sales', 'user_score']:
    converted = pd.to_numeric(df[column], errors='coerce')
    invalid_mask = df[column].notna() & converted.isna()
    print(f'{column}:')
    print(df.loc[invalid_mask, column].value_counts(), end='\n\n')

eu_sales:
eu_sales
unknown    6
Name: count, dtype: int64

jp_sales:
jp_sales
unknown    4
Name: count, dtype: int64

user_score:
user_score
tbd    2464
Name: count, dtype: int64



In [11]:
numeric_columns = [
    'year_of_release', 'na_sales', 'eu_sales', 'jp_sales',
    'other_sales', 'critic_score', 'user_score'
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

df.dtypes

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales           float64
jp_sales           float64
other_sales        float64
critic_score       float64
user_score         float64
rating              object
dtype: object

In [12]:
df[numeric_columns].agg(['min', 'max']).T

,min,max
year_of_release,1980.00,2016.00
na_sales,0.00,41.36
eu_sales,0.00,28.96
jp_sales,0.00,10.22
other_sales,0.00,10.57
critic_score,13.00,98.00
user_score,0.00,9.70


`unknown` в региональных продажах и `tbd` в пользовательских оценках не являются числами. `tbd` означает, что оценка ещё не определена, поэтому такие значения корректно заменить на пропуски. Все потенциально числовые столбцы преобразованы с `errors='coerce'`: некорректные строки стали `NaN`.

Минимальные и максимальные значения после преобразования находятся в допустимых диапазонах. Год пока остаётся вещественным: к целому типу его можно привести после обработки пропусков.

### 2.3. Пропуски

In [13]:
missing_values = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2)
}).sort_values('missing_count', ascending=False)

missing_values

,missing_count,missing_percent
user_score,9268,54.66
critic_score,8714,51.39
rating,6871,40.52
year_of_release,275,1.62
eu_sales,6,0.04
jp_sales,4,0.02
name,2,0.01
genre,2,0.01
platform,0,0.00
na_sales,0,0.00


In [14]:
df[df['name'].isna() | df['genre'].isna()]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
661,NaN,GEN,1993.00,NaN,1.78,0.53,0.00,0.08,NaN,NaN,NaN
14439,NaN,GEN,1993.00,NaN,0.00,0.00,0.03,0.00,NaN,NaN,NaN


Больше всего пропусков содержат оценки пользователей и критиков, а также рейтинг ESRB. Это может быть связано с отсутствием достаточного числа отзывов, более ранним выпуском игры или тем, что игра не проходила оценивание ESRB. Заполнять оценки средними значениями нельзя: это исказит реальные распределения, поэтому они останутся пропусками.

Строки без названия или жанра не позволяют идентифицировать игру. Строки без года нельзя отнести к исследуемому периоду. Их доля мала, поэтому такие записи удалим. Пропуски ESRB заменим индикатором `UNKNOWN`. Пропуски в продажах заполним средним по платформе и году выпуска; на случай отсутствия группового среднего предусмотрим медиану столбца.

In [15]:
initial_rows = len(df)

# Удаляем строки с критичными пропусками.
df = df.dropna(subset=['name', 'genre', 'year_of_release']).copy()
df['year_of_release'] = df['year_of_release'].astype('int64')

# Для отсутствующего ESRB используем явный индикатор.
df['rating'] = df['rating'].fillna('UNKNOWN')

# Заполняем пропуски в продажах средним по платформе и году.
sales_columns = ['na_sales', 'eu_sales', 'jp_sales', 'other_sales']
for column in sales_columns:
    group_mean = df.groupby(['platform', 'year_of_release'])[column].transform('mean')
    df[column] = df[column].fillna(group_mean)
    df[column] = df[column].fillna(df[column].median())

df.isna().sum()

name                  0
platform              0
year_of_release       0
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8594
user_score         9121
rating                0
dtype: int64

После обработки пропуски остались только в оценках пользователей и критиков. Это осознанное решение, позволяющее не подменять отсутствующие оценки искусственными значениями.

### 2.4. Явные и неявные дубликаты

In [16]:
for column in ['platform', 'genre', 'rating']:
    print(f'{column}:')
    print(sorted(df[column].unique()), end='\n\n')

platform:
['2600', '3DO', '3DS', 'DC', 'DS', 'GB', 'GBA', 'GC', 'GEN', 'GG', 'N64', 'NES', 'NG', 'PC', 'PCFX', 'PS', 'PS2', 'PS3', 'PS4', 'PSP', 'PSV', 'SAT', 'SCD', 'SNES', 'TG16', 'WS', 'Wii', 'WiiU', 'X360', 'XB', 'XOne']

genre:
['ACTION', 'ADVENTURE', 'Action', 'Adventure', 'FIGHTING', 'Fighting', 'MISC', 'Misc', 'PLATFORM', 'PUZZLE', 'Platform', 'Puzzle', 'RACING', 'ROLE-PLAYING', 'Racing', 'Role-Playing', 'SHOOTER', 'SIMULATION', 'SPORTS', 'STRATEGY', 'Shooter', 'Simulation', 'Sports', 'Strategy']

rating:
['AO', 'E', 'E10+', 'EC', 'K-A', 'M', 'RP', 'T', 'UNKNOWN']



In [17]:
# Нормализуем регистр и удаляем лишние пробелы.
df['name'] = df['name'].str.strip().str.lower()
df['genre'] = df['genre'].str.strip().str.lower()
df['platform'] = df['platform'].str.strip().str.upper()
df['rating'] = df['rating'].str.strip().str.upper()

for column in ['platform', 'genre', 'rating']:
    print(f'{column}:')
    print(sorted(df[column].unique()), end='\n\n')

platform:
['2600', '3DO', '3DS', 'DC', 'DS', 'GB', 'GBA', 'GC', 'GEN', 'GG', 'N64', 'NES', 'NG', 'PC', 'PCFX', 'PS', 'PS2', 'PS3', 'PS4', 'PSP', 'PSV', 'SAT', 'SCD', 'SNES', 'TG16', 'WII', 'WIIU', 'WS', 'X360', 'XB', 'XONE']

genre:
['action', 'adventure', 'fighting', 'misc', 'platform', 'puzzle', 'racing', 'role-playing', 'shooter', 'simulation', 'sports', 'strategy']

rating:
['AO', 'E', 'E10+', 'EC', 'K-A', 'M', 'RP', 'T', 'UNKNOWN']



In [18]:
duplicates_count = df.duplicated().sum()
print(f'Найдено явных дубликатов: {duplicates_count}')

Найдено явных дубликатов: 235


In [19]:
df = df.drop_duplicates().reset_index(drop=True)

deleted_rows = initial_rows - len(df)
deleted_percent = deleted_rows / initial_rows * 100

print(f'Строк после предобработки: {len(df)}')
print(f'Всего удалено строк: {deleted_rows}')
print(f'Доля удалённых строк: {deleted_percent:.2f}%')
print(f'Оставшихся явных дубликатов: {df.duplicated().sum()}')

Строк после предобработки: 16444
Всего удалено строк: 512
Доля удалённых строк: 3.02%
Оставшихся явных дубликатов: 0


### Вывод по предобработке

- названия столбцов приведены к `snake_case`;
- числовые данные преобразованы к подходящим типам;
- строки без названия, жанра или года удалены;
- пропуски ESRB заменены на `UNKNOWN`;
- пропуски в продажах заполнены с учётом платформы и года выпуска;
- отсутствующие оценки сохранены как `NaN`;
- категориальные значения нормализованы;
- явные дубликаты удалены.

Всего удалено 512 строк, или около 3,02% исходного набора. После предобработки осталось 16 444 строки.

## 3. Фильтрация данных

In [20]:
df_actual = df[df['year_of_release'].between(2000, 2013, inclusive='both')].copy()

print(f'Размер актуального среза: {df_actual.shape}')
print(
    f'Диапазон лет: {df_actual["year_of_release"].min()}–'
    f'{df_actual["year_of_release"].max()}'
)

Размер актуального среза: (12781, 11)
Диапазон лет: 2000–2013


In [21]:
df_actual.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,wii sports,WII,2006,sports,41.36,28.96,3.77,8.45,76.00,8.00,E
2,mario kart wii,WII,2008,racing,15.68,12.76,3.79,3.29,82.00,8.30,E
3,wii sports resort,WII,2009,sports,15.61,10.93,3.28,2.95,80.00,8.00,E
6,new super mario bros.,DS,2006,platform,11.28,9.14,6.50,2.88,89.00,8.50,E
7,wii play,WII,2006,misc,13.96,9.18,2.93,2.84,58.00,6.60,E


В `df_actual` вошла 12 781 запись об играх, выпущенных с 2000 по 2013 год включительно. Исходный очищенный датафрейм `df` сохранён отдельно.

## 4. Категоризация данных

### 4.1. Категории оценок

In [22]:
def categorize_score(score, high_threshold, max_score):
    """Возвращает категорию оценки с заданными границами."""
    if pd.isna(score):
        return 'нет оценки'
    if not 0 <= score <= max_score:
        return 'некорректная оценка'
    if score < 3 * (max_score / 10):
        return 'низкая оценка'
    if score < high_threshold:
        return 'средняя оценка'
    return 'высокая оценка'


df_actual['user_score_category'] = df_actual['user_score'].apply(
    categorize_score,
    args=(8, 10)
)

df_actual['critic_score_category'] = df_actual['critic_score'].apply(
    categorize_score,
    args=(80, 100)
)

In [23]:
category_counts = pd.concat(
    [
        df_actual['user_score_category'].value_counts().rename('user_score'),
        df_actual['critic_score_category'].value_counts().rename('critic_score')
    ],
    axis=1
).fillna(0).astype(int)

category_counts

,user_score,critic_score
нет оценки,6298,5612
средняя оценка,4081,5422
высокая оценка,2286,1692
низкая оценка,116,55


In [24]:
# Проверяем пограничные значения правил категоризации.
boundary_check = pd.DataFrame({
    'user_score': [0, 2.9, 3, 7.9, 8, 10, None],
    'user_category': [
        categorize_score(value, 8, 10)
        for value in [0, 2.9, 3, 7.9, 8, 10, None]
    ],
    'critic_score': [0, 29, 30, 79, 80, 100, None],
    'critic_category': [
        categorize_score(value, 80, 100)
        for value in [0, 29, 30, 79, 80, 100, None]
    ]
})

boundary_check

,user_score,user_category,critic_score,critic_category
0,0.00,низкая оценка,0.00,низкая оценка
1,2.90,низкая оценка,29.00,низкая оценка
2,3.00,средняя оценка,30.00,средняя оценка
3,7.90,средняя оценка,79.00,средняя оценка
4,8.00,высокая оценка,80.00,высокая оценка
5,10.00,высокая оценка,100.00,высокая оценка
6,NaN,нет оценки,NaN,нет оценки


In [25]:
df_actual[
    ['name', 'user_score', 'user_score_category',
     'critic_score', 'critic_score_category']
].head(10)

,name,user_score,user_score_category,critic_score,critic_score_category
0,wii sports,8.00,высокая оценка,76.00,средняя оценка
2,mario kart wii,8.30,высокая оценка,82.00,высокая оценка
3,wii sports resort,8.00,высокая оценка,80.00,высокая оценка
6,new super mario bros.,8.50,высокая оценка,89.00,высокая оценка
7,wii play,6.60,средняя оценка,58.00,средняя оценка
8,new super mario bros. wii,8.40,высокая оценка,87.00,высокая оценка
10,nintendogs,NaN,нет оценки,NaN,нет оценки
11,mario kart ds,8.60,высокая оценка,91.00,высокая оценка
13,wii fit,7.70,средняя оценка,80.00,высокая оценка
14,kinect adventures!,6.30,средняя оценка,61.00,средняя оценка


Для обоих видов оценок добавлена отдельная категория:

- `низкая оценка`: `[0, 3)` для пользователей и `[0, 30)` для критиков;
- `средняя оценка`: `[3, 8)` и `[30, 80)`;
- `высокая оценка`: `[8, 10]` и `[80, 100]`;
- `нет оценки`: исходное значение отсутствовало.

Расчёт вынесен в функцию `categorize_score()`, а отдельная проверка подтверждает правильную обработку границ интервалов.

### 4.2. Топ-7 платформ

In [26]:
top_7_platforms = df_actual['platform'].value_counts().head(7)
top_7_platforms

platform
PS2     2127
DS      2120
WII     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: count, dtype: int64

In [27]:
top_7_platforms_list = top_7_platforms.index.tolist()
df_actual['is_top_7_platform'] = df_actual['platform'].isin(top_7_platforms_list)

df_actual[['name', 'platform', 'is_top_7_platform']].head(10)

,name,platform,is_top_7_platform
0,wii sports,WII,True
2,mario kart wii,WII,True
3,wii sports resort,WII,True
6,new super mario bros.,DS,True
7,wii play,WII,True
8,new super mario bros. wii,WII,True
10,nintendogs,DS,True
11,mario kart ds,DS,True
13,wii fit,WII,True
14,kinect adventures!,X360,True


In [28]:
df_actual.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12781 entries, 0 to 16442
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   name                   12781 non-null  object 
 1   platform               12781 non-null  object 
 2   year_of_release        12781 non-null  int64  
 3   genre                  12781 non-null  object 
 4   na_sales               12781 non-null  float64
 5   eu_sales               12781 non-null  float64
 6   jp_sales               12781 non-null  float64
 7   other_sales            12781 non-null  float64
 8   critic_score           7169 non-null   float64
 9   user_score             6483 non-null   float64
 10  rating                 12781 non-null  object 
 11  user_score_category    12781 non-null  object 
 12  critic_score_category  12781 non-null  object 
 13  is_top_7_platform      12781 non-null  bool   
dtypes: bool(1), float64(6), int64(1), object(6)
memory usage: 1

## 5. Итоговый вывод

В проекте выполнена полная подготовка исторических данных о видеоиграх:

1. Изучена структура исходного набора из 16 956 строк и 11 столбцов.
2. Названия столбцов приведены к `snake_case`.
3. Исправлены типы числовых данных. Значения `unknown` и `tbd` заменены на пропуски при преобразовании.
4. Удалены строки без названия, жанра или года выпуска. Пропуски ESRB обозначены как `UNKNOWN`, а пропуски в продажах заполнены средними по платформе и году.
5. Нормализованы текстовые значения и удалены явные дубликаты. Всего удалено 512 строк (3,02%), осталось 16 444 строки.
6. Сформирован срез `df_actual` за 2000–2013 годы: 12 781 запись.
7. Добавлены поля `user_score_category` и `critic_score_category`, содержащие категории пользовательских и экспертных оценок. Для пропусков сохранена отдельная категория `нет оценки`.
8. Выделен топ-7 платформ по числу игр: **PS2, DS, Wii, PSP, X360, PS3 и GBA**. Поле `is_top_7_platform` показывает принадлежность записи к этому списку.

Полученный датафрейм `df_actual` готов для дальнейшего анализа игровых платформ, жанров, региональных продаж и RPG-игр.